# Standardize output data
Run these cells sequentially

In [1]:
import pluma.schema.outdoor as outdoor

# From utils
import schemas.maat_schema as maat
import schemas.missing_sync as missing_sync
# Pluma-python API
from modules import *

from utils import *
import utils.for_eye_tracker as et
import utils.for_empatica as empatica
import utils.for_climate as climate
import utils.for_setpath as path
import os

/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


In [2]:
# read access only
rawdata = "/mnt/raid/emotional_data_raquel"
# where im adding data to
workdir = "/home/s243636/master-thesis"

# Export lsl, geodata, and empatica streams to CSV

Geodata is the backbone of this process since it is already sampled at 1 Hz and without it, it's impossible to synchronize all the data streams 

In [4]:
# ---------------------------------------------------------------------
# MAIN SCRIPT
datadir    = os.path.join(rawdata, "data")
for participant_folder in os.listdir(datadir):
    if participant_folder.startswith("OE"):
        participant_path = os.path.join(datadir, participant_folder)
        for session_folder in os.listdir(participant_path):
            session_path = os.path.join(participant_path, session_folder)
            session_path = str(session_path)
            if os.path.isdir(session_path):
                session_name = extract_session_name(session_folder)
                # Geodata output
                output = os.path.join(rawdata, 'supp_mine', 'geodata', 'log', f'sub-{participant_folder}', f'ses-{session_name}')
                if not os.path.exists(output): 
                    os.makedirs(output)
                # Empatica output
                csv_outdir = os.path.join(rawdata, 'supp_mine', 'stress_csv', f'sub-{participant_folder}', f'ses-{session_name}')
                if not os.path.exists(csv_outdir): 
                    os.makedirs(csv_outdir)
                # Try to load the dataset using different schemas.
                if "Maat" in session_folder:
                    print("This is a Maat acquisition...")
                    calibrate_ubx_to_harp = True # When true there is better synchronization
                    ubx = True                   # False only for VR acquisitions    
                    dataset    = load_dataset(session_path, schema=maat.build_schema, ubx=ubx, calibrate_ubx_to_harp=calibrate_ubx_to_harp)    
                else:    
                    try:
                        calibrate_ubx_to_harp = True # When true there is better synchronization
                        ubx = True                   # False only for VR acquisitions
                        dataset    = load_dataset(session_path, schema=outdoor.build_schema, ubx=ubx, unity=False, calibrate_ubx_to_harp=calibrate_ubx_to_harp)
                        #plot_summary(dataset, plot_sync_lookup=ubx and calibrate_ubx_to_harp)
                    except:
                        try:
                            print("Trying to load by the missing GPS synchronization schema...")
                            calibrate_ubx_to_harp = True # When true there is better synchronization
                            ubx = True                    # False only for VR acquisitions
                            dataset    = load_dataset(session_path, ubx=ubx, unity=False, calibrate_ubx_to_harp=calibrate_ubx_to_harp, schema=missing_sync.build_schema)
                        except Exception as e:
                            print(f"Error in {participant_folder}-{session_name}")
                            print(e)
                            continue
                
                # Save geodata to csv safely
                has_gps = False

                try:
                    if hasattr(dataset.streams, "TK"):
                        tk_stream = dataset.streams.TK

                        if hasattr(tk_stream, "GPS_HasFix"):
                            gps_fix = tk_stream.GPS_HasFix.data
                            if gps_fix is not None and len(gps_fix) > 0:
                                has_gps = True

                except Exception:
                    has_gps = False


                if has_gps:
                    try:
                        climate.geodata_to_csv(dataset, participant_folder, session_name, output)
                    except Exception as e:
                        print(f"Geodata export failed for {participant_folder}-{session_name}: {e}")
                else:
                    print(f"No valid GPS for {participant_folder}-{session_name}, skipping geodata.")

                # Save empatica data to csv
                try:
                    empatica.empatica_and_ecg_to_csv(dataset, csv_outdir) 
                    print(f'Successfully saved data for {participant_folder}-{session_name}!')
                except:
                    print(f"Error in {participant_folder}-{session_name}")
                    print("No empatica data found...")
                    
                # free memory before next session
                import gc
                del dataset 
                gc.collect() 

/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z/Streams_211 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z/Streams_213 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z/Streams_227 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Har

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
No valid GPS for OE011-Nordhavn, skipping geodata.


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/stream/ubx.py:86: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  reference = self.data["TIM_TM2"].Message[0]
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pandas/core/frame.py:721: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/schema/__init__.py:273: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  utc_offset = self.streams.UBX.positiondata["Time_UTC"][0] - self.geo

Data saved to: /home/s243636/master-thesis/supp/stress_csv/sub-OE011/ses-Nordhavn
Created files:
lsl_markers.csv
ecg_hr.csv
ecg.csv
e4_gsr.csv
e4_temp.csv
e4_ibi.csv
e4_bvp.csv
e4_acc.csv
e4_hr.csv
Successfully saved data for OE011-Nordhavn!


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE015/Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z/Streams_211 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE015/Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z/Streams_213 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE015/Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z/Streams_214 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarnin

...

Data saved to: /home/s243636/master-thesis/supp/stress_csv/sub-OE023/ses-Norreport
Created files:
lsl_markers.csv
ecg_hr.csv
ecg.csv
e4_gsr.csv
e4_temp.csv
e4_ibi.csv
e4_bvp.csv
e4_acc.csv
e4_hr.csv
Successfully saved data for OE023-Norreport!


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE023/Copenhagen_Norrebro_sub-OE201023_2024-06-24T123956Z/Streams_211 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE023/Copenhagen_Norrebro_sub-OE201023_2024-06-24T123956Z/Streams_213 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE023/Copenhagen_Norrebro_sub-OE201023_2024-06-24T123956Z/Streams_227 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Har

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
No valid GPS for OE023-Norrebro, skipping geodata.


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/stream/ubx.py:86: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  reference = self.data["TIM_TM2"].Message[0]
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pandas/core/frame.py:721: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/schema/__init__.py:273: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  utc_offset = self.streams.UBX.positiondata["Time_UTC"][0] - self.geo

Data saved to: /home/s243636/master-thesis/supp/stress_csv/sub-OE023/ses-Norrebro
Created files:
lsl_markers.csv
ecg_hr.csv
ecg.csv
e4_gsr.csv
e4_temp.csv
e4_ibi.csv
e4_bvp.csv
e4_acc.csv
e4_hr.csv
Successfully saved data for OE023-Norrebro!


# Export processed empatica at 1 Hz

In [4]:
# ---------------------------------------------------------------------
# MAIN SCRIPT
datadir = os.path.join(rawdata, 'data')
for participant_folder in os.listdir(datadir):
    if participant_folder.startswith("OE"):
        participant_path = os.path.join(datadir, participant_folder)
        for session_folder in os.listdir(participant_path):
            session_path = os.path.join(participant_path, session_folder)
            if os.path.isdir(session_path):
                session_name = extract_session_name(session_folder)
                input_directory = os.path.join(rawdata, 'supp_mine', 'stress_csv',f'sub-{participant_folder}', f'ses-{session_name}')
                output_directory = os.path.join(input_directory, '_1hz')
                if not os.path.exists(output_directory):
                    os.makedirs(output_directory)

                # Check if BVP file exists
                bvp_file = os.path.join(input_directory, "e4_bvp.csv")

                if not os.path.exists(bvp_file):
                    print(f"No BVP data for {participant_folder}-{session_name}, skipping.")
                    continue

                # Resample Empatica adn ECG data
                try:
                    empatica.export_resampled_empatica_data(input_directory, output_directory)
                except Exception as e:
                    print(f"Error in {participant_folder}-{session_name}: {e}")
                    continue

Loading data...


Processing datetime columns...
Filtering EDA signal...
Processing IBI data...
Resampling data to 1Hz...
Merging data...
Saving combined data to CSV...
Processing complete. All figures and CSV file saved to the output directory.
No BVP data for OE015-Norreport, skipping.
No BVP data for OE019-Hellerup, skipping.
Loading data...
Processing datetime columns...
Filtering EDA signal...
Processing IBI data...
Resampling data to 1Hz...
Merging data...
Saving combined data to CSV...
Processing complete. All figures and CSV file saved to the output directory.
Loading data...


/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/hom

Processing datetime columns...
Filtering EDA signal...
Processing IBI data...
Resampling data to 1Hz...
Merging data...
Saving combined data to CSV...
Processing complete. All figures and CSV file saved to the output directory.
Loading data...
Processing datetime columns...


/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/hom

...

/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/hom

Merging data...
Saving combined data to CSV...
Processing complete. All figures and CSV file saved to the output directory.
Loading data...
Processing datetime columns...
Filtering EDA signal...
Processing IBI data...
Resampling data to 1Hz...
Merging data...


/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/hom

Saving combined data to CSV...
Processing complete. All figures and CSV file saved to the output directory.
Loading data...
Processing datetime columns...
Filtering EDA signal...
Processing IBI data...
Resampling data to 1Hz...
Merging data...
Saving combined data to CSV...
Processing complete. All figures and CSV file saved to the output directory.


/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/home/s243636/master-thesis/notebooks/utils/for_empatica.py:170: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_numeric = df_numeric.set_index(datetime_col).resample(freq).mean().reset_index()
/hom

_________

# Export Eye-tracking metrics

In [4]:
# ---------------------------------------------------------------------
# MAIN SCRIPT
datadir = os.path.join(rawdata, 'data')
for participant_folder in os.listdir(datadir):
    if participant_folder.startswith("OE"):
        participant_path = os.path.join(datadir, participant_folder)
        for session_folder in os.listdir(participant_path):
            session_path = os.path.join(participant_path, session_folder)
            if os.path.isdir(session_path):
                session_name = extract_session_name(session_folder)
                csv_outdir = os.path.join(rawdata, 'supp_mine', 'gaze', f'sub-{participant_folder}', f'ses-{session_name}')
                if not os.path.exists(csv_outdir): 
                    os.makedirs(csv_outdir)
                try: # Try to load the dataset
                    datapicker = create_datapicker(path = session_path, schema=build_schema)
                    dataset    = load_dataset(datapicker.selected_path,schema=build_schema)
                except:
                    try: # Try to load by other schema
                        print("Trying to load by other schema")
                        datapicker = create_datapicker(path=session_path, schema=build_schema, calibrate_ubx_to_harp=False)
                        dataset    = load_dataset(datapicker.selected_path, ubx=True, unity=False, calibrate_ubx_to_harp=False, schema=build_schema)
                    except:
                        print(f"Error in {participant_folder}-{session_name}")
                        continue
                try:
                    # Save the data to csv
                    et.export_gaze_to_csv(dataset, csv_outdir) 
                    print(f'Successfully saved data for {participant_folder}-{session_name}!')
                except Exception as e:
                    print(f"Error in exporting gaze data for {participant_folder}-{session_name}: {e}")
                    continue

/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z/Streams_211 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z/Streams_213 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z/Streams_227 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Har

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/stream/ubx.py:86: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  reference = self.data["TIM_TM2"].Message[0]
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pandas/core/frame.py:721: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/schema/__init__.py:273: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  utc_offset = self.streams.UBX.positiondata["Time_UTC"][0] - self.geo

Successfully saved data for OE011-Nordhavn!


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE015/Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z/Streams_211 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE015/Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z/Streams_213 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE015/Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z/Streams_214 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarnin

...

Successfully saved data for OE023-Norreport!


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE023/Copenhagen_Norrebro_sub-OE201023_2024-06-24T123956Z/Streams_211 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE023/Copenhagen_Norrebro_sub-OE201023_2024-06-24T123956Z/Streams_213 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Harp stream file            @(UNIX) --> /mnt/raid/emotional_data_raquel/data/OE023/Copenhagen_Norrebro_sub-OE201023_2024-06-24T123956Z/Streams_227 could not be found.
  warnings.warn(f"Harp stream file\
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/io/harp.py:43: UserWarning: Har

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/stream/ubx.py:86: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  reference = self.data["TIM_TM2"].Message[0]
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pandas/core/frame.py:721: DeprecationWarning: Passing a BlockManager to GeoDataFrame is deprecated and will raise in a future version. Use public APIs instead.
  warnings.warn(
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/pluma/schema/__init__.py:273: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  utc_offset = self.streams.UBX.positiondata["Time_UTC"][0] - self.geo

Successfully saved data for OE023-Norrebro!


# Add EEG and other informations to the table data
Band power metrics have been computed on preprocessed EEG data using MATLAB. Other data like participant_id, age, sex, and path name are also to be added.
Implemented:
- EEG
- Geodata
- Empatica
- Heart rate
- Eye-tracking

To do:
- Add behavioral and phenotype data

In [3]:
def tidy_data(data, subject_id, session_id, bidsroot):
    """Tidy the tabular data
    Add participant id
    Add session id
    Add age
    Add sex
    Add phenotype
    Add path typology
    Make gps columns (lon,lat,alt) the 2nd, 3rd, and 4th columns
    Rename columns to be more informative
    """
    # Add metadata
    data['participant_id'] = subject_id
    data['session_id'] = session_id
    participants = pd.read_csv(os.path.join(rawdata, 'supp', 'sharedData', 'participant-list_copenhagen.tsv'),sep='\t')
    data['age'] = participants[participants['participant_id'] == subject_id]['age'].values[0]
    data['sex'] = participants[participants['participant_id'] == subject_id]['sex'].values[0]

    # Add behavioral ratings
    subj_num            = int(subject_id[2:])
    id                  = infer_participant_code('copenhagen', subj_num, session_id, 'all')
    sam                 = fetch_beh_ratings(os.path.join(rawdata,'supp'), id)
    if sam.size == 0:  # If no behavioral data, add empty columns
        data['valence']     = np.nan
        data['arousal']     = np.nan
        data['naturalness'] = np.nan
        data['crowdedness'] = np.nan
    else:  # Otherwise, add behavioral ratings
        sam                 = np.tile(sam, (len(data), 1))
        data['valence']     = sam[:, 0]
        data['arousal']     = sam[:, 1]
        data['naturalness'] = sam[:, 2]
        data['crowdedness'] = sam[:, 3]

    # Rename columns
    rename_dict = {
        # Time
        'DateTime': 'time',

        # GPS
        'longitude': 'original_longitude',
        'latitude': 'original_latitude',

        # Climate
        'tk_airquality_iaqindex_value': 'air_quality_index',
        'tk_airquality_temperature_value': 'air_temperature',
        'tk_airquality_humidity_value': 'air_humidity',
        'tk_airquality_airpressure_value': 'air_pressure',
        'tk_soundpressurelevel_spl_value': 'sound_pressure_level',
        'tk_humidity_humidity_value': 'humidity_sensor',
        'tk_analogin_voltage_value': 'analog_voltage',
        'tk_particulatematter_pm1_0_value': 'pm1_0',
        'tk_particulatematter_pm2_5_value': 'pm2_5',
        'tk_particulatematter_pm10_0_value': 'pm10_0',
        'tk_dual0_20ma_solarlight_value': 'solar_light',
        'tk_thermocouple_temperature_value': 'thermocouple_temp',
        'tk_ptc_airtemp_value': 'ptc_air_temp',
        'atmos_northwind_value': 'north_wind',
        'atmos_eastwind_value': 'east_wind',
        'atmos_gustwind_value': 'gust_wind',
        'atmos_airtemperature_value': 'atmospheric_temperature',
        'temp_atmos': 'temp_atmospheric',
        'temp_tk': 'temp_tk',
        'temp_tk_ptc': 'temp_tk_ptc',
        'temp_radiant': 'radiant_temp',
        'hPa': 'Atmosferic Pressure',
        'dew_point': 'Dew Point',

        # Empatica
        'E4_HR_IBI': 'empatica_heart_rate',
        'TEMP': 'skin_temperature',
        'EDA_RAW': 'eda_raw',
        'EDA_PHASIC': 'eda_phasic',
        'AccX': 'x_acceleration',
        'AccY': 'y_acceleration',
        'AccZ': 'z_acceleration',
        'Magnitude': 'acceleration_magnitude',
        'BVP_Values': 'bvp',

        # Other
        'stress_category': 'utci_stress_category'
    }
    data.rename(columns=rename_dict, inplace=True)

    return data

# ---------------------------------------------------------------------
# MAIN SCRIPT
datadir = os.path.join(rawdata, 'data')
for participant_folder in os.listdir(datadir):
    if participant_folder.startswith("OE"):
        participant_path = os.path.join(datadir, participant_folder)
        for session_folder in os.listdir(participant_path):
            session_path = os.path.join(participant_path, session_folder)
            if os.path.isdir(session_path):
                session_name = extract_session_name(session_folder)
                # Define paths of streams
                empatica_data_path = os.path.join(rawdata, 'supp_mine', 'stress_csv', f'sub-{participant_folder}', f'ses-{session_name}', '_1hz', 'data_all_1Hz.csv') 
                geodata_data_path  = os.path.join(rawdata, 'supp', 'geodata', 'log', f'sub-{participant_folder}', f'ses-{session_name}', f'sub-{participant_folder}_ses-{session_name}_geodata.xlsx')
                gaze_data_path = os.path.join(rawdata, 'supp_mine', 'gaze', f'sub-{participant_folder}', f'ses-{session_name}', 'gaze.csv')
                eeg_data_path      = os.path.join(rawdata, 'analysis-climate_physio_eeg_model_pipeline-outdoor_literature', f'sub-{participant_folder}', f'ses-{session_name}', 'data', 'a01_psd_compute.m', 'eeg_power.xlsx')
                # Define output path
                output_path        = os.path.join(rawdata, 'fulldata_mine', f'sub-{participant_folder}', f'ses-{session_name}')
                if not os.path.exists(output_path): 
                    os.makedirs(output_path)
                # Attempt to merge the data (first empatica and geodata, and then eeg)
                # If any file is missing skip to the next data stream
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                #                             GEODATA                           #
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                # Load Geodata
                print("Loading Geodata...")
                try:
                    geodata_data = pd.read_excel(geodata_data_path,parse_dates=['time'])
                except:
                    print("There is no Geodata for this session")
                    continue
                geodata_data = geodata_data.rename(columns={'time':'DateTime'})
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                #                            EMPATICA                           #
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                # Load Empatica data
                print("Loading Empatica data...")
                try:
                    empatica_data = pd.read_csv(empatica_data_path,parse_dates=['DateTime'])
                except:   
                    print(f"There is no Empatica data for this session: sub-{participant_folder}, ses-{session_name}")
                    print("Adding empty Empatica data...")
                    # Create an empty DataFrame with the same column structure
                    empatica_columns = [
                        "DateTime", "E4_HR", "E4_HR_IBI", "TEMP", "EDA_RAW",
                        "EDA_PHASIC", "AccX", "AccY", "AccZ", "Magnitude", "BVP_Values"
                    ]
                    empatica_data = pd.DataFrame(columns=empatica_columns)
                    empatica_data['DateTime'] = geodata_data['DateTime']
                # Merge Empatica data with climate data on 'DateTime'
                print("Merging Empatica data with climate data...")
                # Sort (required for merge_asof)
                # Ensure datetime format
                empatica_data['DateTime'] = pd.to_datetime(empatica_data['DateTime'], errors='coerce')
                geodata_data['DateTime']  = pd.to_datetime(geodata_data['DateTime'], errors='coerce')

                empatica_data = empatica_data.sort_values("DateTime")
                geodata_data  = geodata_data.sort_values("DateTime")
                # cahnge from how=inner to how=left bc no geodata as of yet
                # combined_data = pd.merge_asof(empatica_data,geodata_data,on="DateTime",direction="nearest",tolerance=pd.Timedelta("2s"))
                combined_data = pd.merge(empatica_data, geodata_data, on='DateTime', how='inner')
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                #                            GAZE                               #
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                print("Loading Gaze data...")
                try:
                    gaze_data = pd.read_csv(gaze_data_path)
                    gaze_data['Seconds'] = pd.to_datetime(gaze_data['Seconds'], errors='coerce')
                    gaze_data = gaze_data.rename(columns={'Seconds': 'DateTime'})
                    gaze_data = gaze_data[['DateTime','GazeX','GazeY']]
                except:
                    print(f"There is no Gaze data for this session: sub-{participant_folder}, ses-{session_name}")
                    print("Adding empty Gaze data...")
                    gaze_data = pd.DataFrame(columns=["DateTime","GazeX","GazeY"])

                print("Merging Gaze data...")

                # ensure both sides are datetime
                combined_data['DateTime'] = pd.to_datetime(combined_data['DateTime'], errors='coerce')
                gaze_data['DateTime'] = pd.to_datetime(gaze_data['DateTime'], errors='coerce')
                combined_data = combined_data.sort_values("DateTime")
                gaze_data = gaze_data.sort_values("DateTime")
                combined_data = pd.merge_asof(combined_data,gaze_data,on="DateTime",direction="nearest",tolerance=pd.Timedelta("1s"))
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                #                            EEG                                #
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                # Load EEG data
                print("Loading EEG data...")
                try:
                    eeg_data = pd.read_excel(eeg_data_path)
                except:
                    print(f"There is no EEG data for this session: sub-{participant_folder}, ses-{session_name}")
                    print("Adding empty EEG data...")
                    # Create an empty DataFrame with the same column structure
                    eeg_columns = [
                        "Time", "delta", "theta", "alpha", "beta", "gamma",
                        "frontal alpha", "frontal midline theta", "theta-beta ratio",	
                        "frontal alpha asymmetry", "frontal theta"
                    ]
                    eeg_data = pd.DataFrame(columns=eeg_columns)
                eeg_data = eeg_data.rename(columns={'Time':'DateTime'})
                if eeg_data.empty:
                    eeg_data['DateTime'] = combined_data['DateTime']
                    # Merge EEG data with combined data on 'DateTime'
                    print("Merging empty EEG data with combined data...")
                    combined_data = pd.merge(combined_data, eeg_data, on='DateTime', how='left')
                else:
                    # Merge EEG data with combined data on 'DateTime'
                    print("Merging EEG data with combined data...")
                    combined_data = pd.merge(combined_data, eeg_data, on='DateTime', how='left')
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                #                            TIDY                               #
                # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
                # Tidy combined data
                print("Tidying combined data...")
                combined_data = tidy_data(combined_data, participant_folder, session_name, bidsroot)
                # Export combined data
                print("Exporting combined data...")
                combined_data.to_csv(os.path.join(output_path, 'alldata.csv'), index=False)

Loading Geodata...
Loading Empatica data...
Merging Empatica data with climate data...
Loading Gaze data...
Merging Gaze data...
Loading EEG data...
Merging EEG data with combined data...
Tidying combined data...
Exporting combined data...
Loading Geodata...
Loading Empatica data...
There is no Empatica data for this session: sub-OE015, ses-Norreport
Adding empty Empatica data...
Merging Empatica data with climate data...
Loading Gaze data...
Merging Gaze data...
Loading EEG data...
There is no EEG data for this session: sub-OE015, ses-Norreport
Adding empty EEG data...
Merging empty EEG data with combined data...
Tidying combined data...
Exporting combined data...
Loading Geodata...
Loading Empatica data...
There is no Empatica data for this session: sub-OE019, ses-Hellerup
Adding empty Empatica data...
Merging Empatica data with climate data...
Loading Gaze data...
Merging Gaze data...
Loading EEG data...
Merging empty EEG data with combined data...
Tidying combined data...
Exporting

In [6]:
root = os.path.join(rawdata, "fulldata_mine")
results = []

for sub in os.listdir(root):
    sub_path = os.path.join(root, sub)

    if not os.path.isdir(sub_path):
        continue

    for ses in os.listdir(sub_path):
        ses_path = os.path.join(sub_path, ses)
        file_path = os.path.join(ses_path, "alldata.csv")

        if not os.path.exists(file_path):
            continue

        df = pd.read_csv(file_path)

        # ----- Sensor checks -----

        # Empatica
        empatica_cols = ["empatica_heart_rate", "eda_raw", "skin_temperature"]
        empatica = any(col in df.columns and df[col].notna().any() for col in empatica_cols)

        # Geodata
        geo_cols = ["longitude_corrected", "latitude_corrected"]
        geodata = any(col in df.columns and df[col].notna().any() for col in geo_cols)

        # Gaze
        gaze_cols = ["GazeX", "GazeY"]
        gaze = any(col in df.columns and df[col].notna().any() for col in gaze_cols)

        # EEG
        eeg_cols = ["delta", "theta", "alpha", "beta", "gamma"]
        eeg = any(col in df.columns and df[col].notna().any() for col in eeg_cols)

        results.append({"participant": sub, "session": ses, "Empatica": empatica, "Geodata (*_correct)": geodata, "Gaze": gaze, "EEG": eeg})

summary = pd.DataFrame(results)
summary.to_csv(os.path.join(rawdata, "fulldata_mine/dataset_sensor_availability.csv"), index=False)
summary

,participant,session,Empatica,Geodata (*_correct),Gaze,EEG
0,sub-OE004,ses-Norreport,True,True,True,True
1,sub-OE018,ses-Hellerup,True,True,True,True
2,sub-OE024,ses-Nordhavn,True,False,True,False
3,sub-OE019,ses-Hellerup,False,True,True,False
4,sub-OE011,ses-Nordhavn,True,False,True,True
5,sub-OE023,ses-Norreport,True,True,True,True
6,sub-OE023,ses-Norrebro,True,True,True,True
7,sub-OE023,ses-Nordhavn,True,True,True,False
8,sub-OE010,ses-Nordhavn,True,True,True,False
9,sub-OE020,ses-Norrebro,True,False,True,True


In [64]:
# Load your summary
summary_path = os.path.join(rawdata, "fulldata_mine/dataset_sensor_availability.csv")
summary = pd.read_csv(summary_path)

# Exclusions
exclude = [
    ("sub-OE015", "ses-Norreport"),
    ("sub-OE007", "ses-Hellerup"),
    ("sub-OE012", "ses-Norreport"),
    ("sub-OE017", "ses-Norreport"),
]

# Filter participants WITHOUT EEG (and not excluded)
no_eeg = summary[
    (summary["EEG"] == False) &
    (~summary[["participant", "session"]].apply(tuple, axis=1).isin(exclude))
]

print(f"\nFound {len(no_eeg)} sessions without EEG (without taking into account participants that have no EEG files, that is\n"
      "OE015-Norreport, OE007-Hellerup, OE012-Norreport and OE017-Norreport):\n")

# Loop through them
for _, row in no_eeg.iterrows():
    sub = row["participant"]
    ses = row["session"]

    print("\n" + "="*50)
    print(f"{sub} | {ses}")

    # Build EEG path (FIX THIS IF NEEDED)
    eeg_path = os.path.join(
        rawdata,
        "analysis-climate_physio_eeg_model_pipeline-outdoor_literature",
        sub,
        ses,
        "data",
        "a01_psd_compute.m", 
        "eeg_power.xlsx"
    )

    print("Path:", eeg_path)
    print("Exists:", os.path.exists(eeg_path))

    if not os.path.exists(eeg_path):
        print("❌ File missing")
        continue

    try:
        eeg_df = pd.read_excel(eeg_path)

        print("Shape:", eeg_df.shape)
        print("Columns:", list(eeg_df.columns))

        if eeg_df.empty:
            print("⚠️ File exists but is EMPTY")
        else:
            print("✅ File has data, preview:")
            print(eeg_df.head())

            # Check if EEG columns are actually NaN
            eeg_cols = ["delta", "theta", "alpha", "beta", "gamma"]
            existing = [c for c in eeg_cols if c in eeg_df.columns]

            if existing:
                all_nan = eeg_df[existing].isna().all().all()
                print("All EEG values NaN:", all_nan)
            else:
                print("⚠️ Expected EEG columns not found")

    except Exception as e:
        print("❌ Error reading file:", e)


Found 4 sessions without EEG (without taking into account participants that have no EEG files, that is
OE015-Norreport, OE007-Hellerup, OE012-Norreport and OE017-Norreport):


sub-OE024 | ses-Nordhavn
Path: /mnt/raid/emotional_data_raquel/analysis-climate_physio_eeg_model_pipeline-outdoor_literature/sub-OE024/ses-Nordhavn/data/a01_psd_compute.m/eeg_power.xlsx
Exists: True
Shape: (0, 11)
Columns: ['Time', 'delta', 'theta', 'alpha', 'beta', 'gamma', 'frontal alpha', 'frontal midline theta', 'theta-beta ratio', 'frontal alpha asymmetry', 'frontal theta']
⚠️ File exists but is EMPTY

sub-OE019 | ses-Hellerup
Path: /mnt/raid/emotional_data_raquel/analysis-climate_physio_eeg_model_pipeline-outdoor_literature/sub-OE019/ses-Hellerup/data/a01_psd_compute.m/eeg_power.xlsx
Exists: True
Shape: (0, 11)
Columns: ['Time', 'delta', 'theta', 'alpha', 'beta', 'gamma', 'frontal alpha', 'frontal midline theta', 'theta-beta ratio', 'frontal alpha asymmetry', 'frontal theta']
⚠️ File exists but is EMPTY

s

### Missing data summary

- **EEG:** 
  - Missing (no files at all) for:
    - sub-OE015 — ses-Norreport
    - sub-OE007 — ses-Hellerup
    - sub-OE012 — ses-Norreport
    - sub-OE017 — ses-Norreport
  - Present but empty (no usable data) for:
    - sub-OE024 — ses-Nordhavn
    - sub-OE019 — ses-Hellerup
    - sub-OE023 — ses-Nordhavn
    - sub-OE010 — ses-Nordhavn

- **Empatica (physiological data) missing for:**
  - sub-OE015 — ses-Norreport  
  - sub-OE019 — ses-Hellerup  
  - sub-OE007 — ses-Hellerup  
  - sub-OE012 — ses-Norreport  
  - sub-OE017 — ses-Norreport  

- **Geodata (environmental sensors) missing for:**
  - sub-OE012 — ses-Norreport
  - sub-OE017 — ses-Norreport
  - sub-OE007 — ses-Hellerup 

- **Gaze (eye-tracking) missing after alignment for:**
  - sub-OE012 — ses-Norreport
  - sub-OE017 — ses-Norreport
  - sub-OE007 — ses-Hellerup 

- All other sessions contain valid **EEG + Empatica + Geodata + Gaze** data.

## Tidy geodata

In [51]:
def tidy_data(data, subject_id, session_id, bidsroot):
    """Tidy the tabular data
    Add participant id
    Add session id
    Add age
    Add sex
    Add phenotype
    Add path typology
    Make gps columns (lon,lat,alt) the 2nd, 3rd, and 4th columns
    Rename columns to be more informative
    """
    # Add metadata
    data['participant_id'] = subject_id
    data['session_id'] = session_id
    participants = pd.read_csv(os.path.join(rawdata, 'supp', 'sharedData', 'participant-list_copenhagen.tsv'),sep='\t')
    data['age'] = participants[participants['participant_id'] == subject_id]['age'].values[0]
    data['sex'] = participants[participants['participant_id'] == subject_id]['sex'].values[0]

    # Add behavioral ratings
    subj_num            = int(subject_id[2:])
    id                  = infer_participant_code('copenhagen', subj_num, session_id, 'all')
    sam                 = fetch_beh_ratings(os.path.join(rawdata,'supp'), id)
    if sam.size == 0:  # If no behavioral data, add empty columns
        data['valence']     = np.nan
        data['arousal']     = np.nan
        data['naturalness'] = np.nan
        data['crowdedness'] = np.nan
    else:  # Otherwise, add behavioral ratings
        sam                 = np.tile(sam, (len(data), 1))
        data['valence']     = sam[:, 0]
        data['arousal']     = sam[:, 1]
        data['naturalness'] = sam[:, 2]
        data['crowdedness'] = sam[:, 3]

    # Rename columns
    rename_dict = {
        # Time
        'DateTime': 'time',

        # GPS
        'longitude': 'original_longitude',
        'latitude': 'original_latitude',

        # Climate
        'tk_airquality_iaqindex_value': 'air_quality_index',
        'tk_airquality_temperature_value': 'air_temperature',
        'tk_airquality_humidity_value': 'air_humidity',
        'tk_airquality_airpressure_value': 'air_pressure',
        'tk_soundpressurelevel_spl_value': 'sound_pressure_level',
        'tk_humidity_humidity_value': 'humidity_sensor',
        'tk_analogin_voltage_value': 'analog_voltage',
        'tk_particulatematter_pm1_0_value': 'pm1_0',
        'tk_particulatematter_pm2_5_value': 'pm2_5',
        'tk_particulatematter_pm10_0_value': 'pm10_0',
        'tk_dual0_20ma_solarlight_value': 'solar_light',
        'tk_thermocouple_temperature_value': 'thermocouple_temp',
        'tk_ptc_airtemp_value': 'ptc_air_temp',
        'atmos_northwind_value': 'north_wind',
        'atmos_eastwind_value': 'east_wind',
        'atmos_gustwind_value': 'gust_wind',
        'atmos_airtemperature_value': 'atmospheric_temperature',
        'temp_atmos': 'temp_atmospheric',
        'temp_tk': 'temp_tk',
        'temp_tk_ptc': 'temp_tk_ptc',
        'temp_radiant': 'radiant_temp',
        'hPa': 'Atmosferic Pressure',
        'dew_point': 'Dew Point',

        # Empatica
        'E4_HR_IBI': 'empatica_heart_rate',
        'TEMP': 'skin_temperature',
        'EDA_RAW': 'eda_raw',
        'EDA_PHASIC': 'eda_phasic',
        'AccX': 'x_acceleration',
        'AccY': 'y_acceleration',
        'AccZ': 'z_acceleration',
        'Magnitude': 'acceleration_magnitude',
        'BVP_Values': 'bvp',

        # Other
        'stress_category': 'utci_stress_category'
    }
    data.rename(columns=rename_dict, inplace=True)

    # Dynamically reorder columns
    column_order = [
        'time',                                                    # Time-related column
        'longitude_corrected', 'latitude_corrected', 'cum_dist',   # Corrercted GPS columns
        'original_longitude', 'original_latitude', 'elevation',    # Original GPS columns
        'day_of_year', 'day', 'month', 'hour', 'minute', 'second', # Time-related columns
        'participant_id', 'session_id', 'age', 'sex',              # Metadata columns
        'utci_stress_category',                                    # Stress category
        'typology',                                                # Path typology
        'valence', 'arousal', 'naturalness', 'crowdedness',        # Behavioral ratings
        # Climate
        'air_quality_index', 'air_temperature', 'air_humidity', 'air_pressure', 'sound_pressure_level',
        'humidity_sensor', 'analog_voltage', 'pm1_0', 'pm2_5', 'pm10_0', 'solar_light', 'thermocouple_temp',
        'ptc_air_temp', 'north_wind', 'east_wind', 'gust_wind', 'atmospheric_temperature', 'temp_atmospheric',
        'temp_tk', 'temp_tk_ptc', 'radiant_temp', 'Atmosferic Pressure', 'Dew Point',
        # Empatica
        'empatica_heart_rate', 'skin_temperature', 'eda_raw', 'eda_phasic', 'x_acceleration', 'y_acceleration', 'z_acceleration', 'acceleration_magnitude', 'bvp',
        # EEG
        'delta', 'theta', 'alpha', 'beta', 'gamma', 'frontal alpha', 'frontal midline theta', 'theta-beta ratio', 'frontal alpha asymmetry', 'frontal theta'
    ]
    other_columns = [col for col in data.columns if col not in column_order]
    reordered_columns = column_order + other_columns

    # Apply the reordered column structure
    data = data[reordered_columns]

    return data

def process_single_participant(participant_id, session_name):
    """
    Process data for a single participant and session.
    
    Parameters:
    -----------
    participant_id : str
        Participant ID (e.g., "OE01")
    session_name : str
        Session name/number (e.g., "1")
        
    Returns:
    --------
    pd.DataFrame
        Combined and processed data for the participant
    """
    # Define paths for the participant
    empatica_data_path = os.path.join(
        rawdata, 'supp_mine', 'stress_csv', 
        f'sub-{participant_id}', f'ses-{session_name}', 
        '_1hz', 'data_all_1Hz.csv'
    )
    
    geodata_data_path = os.path.join(
        rawdata, 'supp', 'geodata', 'log',
        f'sub-{participant_id}', f'ses-{session_name}',
        f'sub-{participant_id}_ses-{session_name}_geodata.xlsx'
    )
    
    eeg_data_path = os.path.join(
        rawdata, 'analysis-climate_physio_eeg_model_pipeline-outdoor_literature',
        f'sub-{participant_id}', f'ses-{session_name}',
        'data', 'a01_psd_compute.m', 'eeg_power.xlsx'
    )
    
    output_path = os.path.join(
        rawdata, 'fulldata_mine',
        f'sub-{participant_id}', f'ses-{session_name}'
    )
    
    # Create output directory if it doesn't exist
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    # Load Geodata
    print("Loading Geodata...")
    try:
        geodata_data = pd.read_excel(geodata_data_path, parse_dates=['time'])
        geodata_data = geodata_data.rename(columns={'time': 'DateTime'})
    except Exception as e:
        print(f"Error loading Geodata: {e}")
        return None

    # Load Empatica data
    print("Loading Empatica data...")
    try:
        empatica_data = pd.read_csv(empatica_data_path, parse_dates=['DateTime'])
    except Exception as e:
        print("Adding empty Empatica data...")
        empatica_columns = [
            "DateTime", "E4_HR", "E4_HR_IBI", "TEMP", "EDA_RAW",
            "EDA_PHASIC", "AccX", "AccY", "AccZ", "Magnitude", "BVP_Values"
        ]
        empatica_data = pd.DataFrame(columns=empatica_columns)
        empatica_data['DateTime'] = geodata_data['DateTime']

    # Merge Empatica and Geodata
    print("Merging Empatica data with climate data...")
    combined_data = pd.merge(empatica_data, geodata_data, on='DateTime', how='inner')

    # Load EEG data
    print("Loading EEG data...")
    try:
        eeg_data = pd.read_excel(eeg_data_path)
        eeg_data = eeg_data.rename(columns={'Time': 'DateTime'})
    except Exception as e:
        print("Adding empty EEG data...")
        eeg_columns = [
            "DateTime", "delta", "theta", "alpha", "beta", "gamma",
            "frontal alpha", "frontal midline theta", "theta-beta ratio",
            "frontal alpha asymmetry", "frontal theta"
        ]
        eeg_data = pd.DataFrame(columns=eeg_columns)
        eeg_data['DateTime'] = combined_data['DateTime']

    # Merge EEG data
    print("Merging EEG data with combined data...")
    if eeg_data.empty:
        combined_data = pd.merge(combined_data, eeg_data, on='DateTime', how='left')
    else:
        combined_data = pd.merge(combined_data, eeg_data, on='DateTime', how='inner')

    # Tidy combined data
    print("Tidying combined data...")
    combined_data = tidy_data(combined_data, participant_id, session_name, bidsroot)

    # Export combined data
    print("Exporting combined data...")
    output_file = os.path.join(output_path, 'alldata.csv')
    combined_data.to_csv(output_file, index=False)
    print(f"Data saved to: {output_file}")

    return combined_data

# Example usage:
# if __name__ == "__main__":
#     # Replace these with your specific participant and session
#     participant_id = "OE016"  # Example participant
#     session_name = "Graca"       # Example session
    
#     data = process_single_participant(participant_id, session_name)
#     if data is not None:
#         print("Processing completed successfully!")

In [52]:
datadir = os.path.join(rawdata, "data")

for participant_folder in os.listdir(datadir):
    if participant_folder.startswith("OE"):
        
        participant_path = os.path.join(datadir, participant_folder)
        
        for session_folder in os.listdir(participant_path):
            session_path = os.path.join(participant_path, session_folder)
            
            if os.path.isdir(session_path):
                
                session_name = extract_session_name(session_folder)
                
                print(f"\nProcessing sub-{participant_folder} | ses-{session_name}")
                
                try:
                    process_single_participant(participant_folder, session_name)
                except Exception as e:
                    print(f"❌ Error for sub-{participant_folder} ses-{session_name}: {e}")
                    continue


Processing sub-OE011 | ses-Nordhavn
Loading Geodata...
Loading Empatica data...
Merging Empatica data with climate data...
Loading EEG data...
Merging EEG data with combined data...
Tidying combined data...
❌ Error for sub-OE011 ses-Nordhavn: "['longitude_corrected', 'latitude_corrected', 'cum_dist', 'typology'] not in index"

Processing sub-OE015 | ses-Norreport
Loading Geodata...
Loading Empatica data...
Adding empty Empatica data...
Merging Empatica data with climate data...
Loading EEG data...
Adding empty EEG data...
Merging EEG data with combined data...
Tidying combined data...
❌ Error for sub-OE015 ses-Norreport: "['typology'] not in index"

Processing sub-OE019 | ses-Hellerup
Loading Geodata...
Loading Empatica data...
Adding empty Empatica data...
Merging Empatica data with climate data...
Loading EEG data...
Merging EEG data with combined data...
Tidying combined data...
❌ Error for sub-OE019 ses-Hellerup: "['typology'] not in index"

Processing sub-OE010 | ses-Nordhavn
Loa